# Kidney Perturbation Analysis

- Creator: Sebastian Birk (sb75@sanger.ac.uk)
- Date of Creation: 28.01.2026
- Date of Last Modification: 29.01.2026 (by Sebastian Birk; sb75@sanger.ac.uk)

# 1. Setup

## 1.1 Load Libraries

In [1]:
%load_ext autoreload
%autoreload 2

In [31]:
import os
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from datasets import load_from_disk

from app.infer import embed_dataset, perturb_dataset, tokenize_adata
from nichejepa.preprocessors import construct_neighbor_graph

## 1.2 Define Config/Params

In [48]:
# Define paths
#model_folder_path = '/nfs/team361/sb75/nichejepa-reproducibility/artifacts/models/11012026_190057_836' # cross-tissue model
model_folder_path = '/nfs/team361/sb75/nichejepa-reproducibility/artifacts/models/01112025_081731_614' # benchmarking model
gene_ensembl_map_file_path = '/nfs/team361/sb75/DATASETS/genes/homo_sapiens_gene_name_to_ensembl_id_dict.pkl'
kidney_perturbation_folder_path = '/nfs/team361/sb75/DATASETS/gold/cell-graph-tokenizer/kidney_perturbation'
cache_directory_path = f'{kidney_perturbation_folder_path}/cache'
results_directory_path = f'{kidney_perturbation_folder_path}/results'

In [44]:
niche_key = "NEMO_neighborhood_annotation"
batch_key = 'Study_slice_id'

emb_layer = None
batch_size = 128
pin_memory = False
num_workers = 12
agg_excluded_tokens = None
top_k = None

## 1.3 Run Notebook Setup

In [8]:
os.makedirs(f'{results_directory_path}/figures', exist_ok=True)

In [9]:
plt.rcParams['font.size'] = 5
plt.rcParams['text.usetex'] = False
plt.rcParams['svg.fonttype'] = 'none'

In [10]:
# Improve reproducibility of UMAP and Leiden
os.environ['NUMBA_CPU_NAME'] = 'generic'

In [11]:
sc.settings.figdir = f'{results_directory_path}/figures'

sc.settings.set_figure_params(
    dpi=300,
    dpi_save=300,
    figsize=(6, 4),
    facecolor='white',
    fontsize=10
)

# 2. Analysis

## 2.1 Load Data

In [12]:
# Load original raw data
adata = sc.read_h5ad(
    "/nfs/team361/dj17/NEMO_DJ/KidneyToxicity/NEMO_UpdatedTokenisationandEmbedding_November22nd25/Labeltransfer/forshare/nemokidneyxeniumatlas_annotated.h5ad")
adata

AnnData object with n_obs × n_vars = 1517894 × 4949
    obs: 'cell_id', 'Study_slice_id', 'NEMO_K_SLICE_ID', 'NEMO_K_PATIENT_ID', 'Corpus_ID', 'Study_patient_id', 'Data_source', 'Platform', 'Tissue_subregion', 'Status', 'Diagnosis', 'Type', 'Age', 'Age_grouping', 'Sex', 'eGFR', 'eGFR_group', 'BMI', 'Obesity_status', 'eGFR baseline', 'eGFR change', 'RCC_Treatment', 'batch', 'NEMO_cell_label', 'NEMO_neighborhood_label', 'tech', 'scANVI_pred', 'scANVI_confidence', 'slice', 'NEMO_neighborhood_annotation', 'leiden', 'highlight_neigh'
    uns: 'Data_source_colors', 'NEMO_K_PATIENT_ID_colors', 'NEMO_cell_label_colors', 'NEMO_neighborhood_annotation_colors', 'NEMO_neighborhood_label_colors', 'Obesity_status_colors', 'Sex_colors', 'eGFR_group_colors', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'rank_genes_groups', 'scANVI_pred_colors', 'umap'
    obsm: 'X_umap', 'cell_emb', 'cell_umap', 'neighborhood_emb', 'neighborhood_umap', 'spatial'
    layers: 'counts', 'lognorm'
    obsp: 'connectiv

In [13]:
# Check columns
adata.obs.head(1)

,cell_id,Study_slice_id,NEMO_K_SLICE_ID,NEMO_K_PATIENT_ID,Corpus_ID,Study_patient_id,Data_source,Platform,Tissue_subregion,Status,...,batch,NEMO_cell_label,NEMO_neighborhood_label,tech,scANVI_pred,scANVI_confidence,slice,NEMO_neighborhood_annotation,leiden,highlight_neigh
CV9-KID-0-FT-5-S4_1007_batch22_0-slice0,1007_batch22_0,CV9-KID-0-FT-5-S4,K002,P002,xhk1007-CV9-41b_1p,CV9,DIAMOND,Xenium 5k,na,Healthy,...,CV9-KID-0-FT-5-S4_with_emb_NEMO_common_qc,72,4,Xenium 5k,Vascular mural cell,0.485031,0,Distal nephron neighborhood,13,0


In [14]:
# List samples
# CV*** -> pre-treatment
# DI*** -> post-treatment
adata.obs[['Study_slice_id', 'Data_source']].drop_duplicates()

,Study_slice_id,Data_source
CV9-KID-0-FT-5-S4_1007_batch22_0-slice0,CV9-KID-0-FT-5-S4,DIAMOND
CV1-KID-0-FT-5-S6_aaaajpak-1-slice1,CV1-KID-0-FT-5-S6,DIAMOND
CV7-KID-0-FT-5-S5_aaabiano-1-slice2,CV7-KID-0-FT-5-S5,DIAMOND
DI15-KID-0-FT-3_aaaaifma-1-slice3,DI15-KID-0-FT-3,ACED
CV1-KID-0-FT-5-S5_aaachdbk-1-slice4,CV1-KID-0-FT-5-S5,DIAMOND
DI13-KID-0-FT-3_aaabbfck-1-slice5,DI13-KID-0-FT-3,ACED
CV9-KID-0-FT-5_aaaalbcc-1-slice6,CV9-KID-0-FT-5,DIAMOND
DI14-KID-0-FT-2_aaabhged-1-slice7,DI14-KID-0-FT-2,ACED
CV5-KID-0-FT-5_aaaabcmf-1-slice8,CV5-KID-0-FT-5,DIAMOND
CV6-KID-0-FT-5-S3_1007_batch24_0-slice9,CV6-KID-0-FT-5-S3,DIAMOND


In [15]:
# Compute sample-wise neighbor graph, concatenate, and store in adata.obsp['spatial_connectivities']
adata = construct_neighbor_graph(
    adata,
    n_neighs=10,
    radius=None,
    delaunay=False,
    include_self_loop=False,
    compute_neighbor_counts=False,
    batch_key=batch_key
)

# Compute raw neighborhood counts
adata.layers['counts_neighborhood'] = (
    adata.obsp['spatial_connectivities'].T @ adata.layers['counts'])

# Compute lognorm neighborhood counts (performing same normalization as on raw counts)
adata.X = adata.layers["counts_neighborhood"].copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.layers["lognorm_neighborhood"] = adata.X.copy()
adata.X = adata.layers["lognorm"].copy()

/nfs/team361/sb75/nichejepa/src/nichejepa/preprocessors/graph.py:109: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  first_idx = adata.obs.reset_index().groupby(batch_key).head(1).index



Processing batch CV9-KID-0-FT-5-S4...
Loading data...
Computing spatial neighborhood graph...


/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/squidpy/gr/_utils.py:203: ImplicitModificationWarning: Setting element `.obsp['spatial_connectivities']` of view, initializing view as actual.
  obj[key] = data



Processing batch CV1-KID-0-FT-5-S6...
Loading data...
Computing spatial neighborhood graph...


/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/squidpy/gr/_utils.py:203: ImplicitModificationWarning: Setting element `.obsp['spatial_connectivities']` of view, initializing view as actual.
  obj[key] = data



Processing batch CV7-KID-0-FT-5-S5...
Loading data...
Computing spatial neighborhood graph...


... (22 outputs left out)

/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/squidpy/gr/_utils.py:203: ImplicitModificationWarning: Setting element `.obsp['spatial_connectivities']` of view, initializing view as actual.
  obj[key] = data



Processing batch DI16-KID-0-FT-1...
Loading data...
Computing spatial neighborhood graph...


/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/squidpy/gr/_utils.py:203: ImplicitModificationWarning: Setting element `.obsp['spatial_connectivities']` of view, initializing view as actual.
  obj[key] = data


/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/legacy_api_wrap/__init__.py:82: UserWarning: Some cells have zero counts
  return fn(*args_all, **kw)


In [16]:
# Validate that everything is ok
print(adata.obsp['spatial_connectivities'].sum(1))
n = 30
print(adata.layers['counts_neighborhood'][:30].toarray())
print(adata.layers['lognorm_neighborhood'][:30].toarray())
print(adata.layers['counts'][:30].toarray())
print(adata.layers['lognorm'][:30].toarray())
print(adata.X[:30].toarray())

[[10.]
 [10.]
 [10.]
 ...
 [10.]
 [10.]
 [10.]]
[[ 0.  5.  0. ...  0.  0.  0.]
 [ 0. 10.  0. ...  0.  0.  2.]
 [ 0. 10.  0. ...  0.  0.  4.]
 ...
 [ 0. 11.  0. ...  0.  1.  1.]
 [ 0. 11.  0. ...  0.  1.  1.]
 [ 0.  9.  0. ...  0.  0.  1.]]
[[0.         2.46250942 0.         ... 0.         0.         0.        ]
 [0.         2.89606595 0.         ... 0.         0.         1.48626583]
 [0.         2.91599883 0.         ... 0.         0.         2.07780264]
 ...
 [0.         3.18158902 0.         ... 0.         1.13096198 1.13096198]
 [0.         2.88188638 0.         ... 0.         0.92886262 0.92886262]
 [0.         3.14818648 0.         ... 0.         0.         1.24619486]]
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 2.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 1. 0. ... 0. 0. 0.]
 [0. 1. 0. ... 0. 1. 0.]]
[[0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         3.58765993]
 [0.         0

In [20]:
# Add ensembl IDs to AnnData object
with open(gene_ensembl_map_file_path, "rb") as f:
    gene_ensembl_map_dict = pickle.load(f)

gene_names = [gene_name for gene_name in adata.var_names.tolist()]
adata.var.index = gene_names

harmonized_gene_names = []
matching_ensembl_ids = []
for gene_name in gene_names:
    if gene_name in gene_ensembl_map_dict.keys():
        harmonized_gene_names.append(gene_name)
        matching_ensembl_ids.append(gene_ensembl_map_dict[gene_name])

adata.var['ensembl_id'] = matching_ensembl_ids
adata.var

,ensembl_id
A2ML1,ENSG00000166535
AAMP,ENSG00000127837
AAR2,ENSG00000131043
AARSD1,ENSG00000266967
ABAT,ENSG00000183044
...,...
ZSCAN26,ENSG00000197062
ZSWIM6,ENSG00000130449
ZUP1,ENSG00000153975
ZYG11B,ENSG00000162378


In [21]:
adata.layers

Layers with keys: counts, lognorm, counts_neighborhood, lognorm_neighborhood

In [23]:
# Write AnnData to disk
adata.write(f"{kidney_perturbation_folder_path}/nemokidneyxeniumatlas_annotated_processed.h5ad")

In [49]:
# Load processed AnnData
adata = sc.read_h5ad(f"{kidney_perturbation_folder_path}/nemokidneyxeniumatlas_annotated_processed.h5ad")

## 2.2 Compute Differential Genes Between Pre- and Post-treatment Conditions

Check https://docs.google.com/spreadsheets/d/1i6p3QRTE5Lh32AejWE6Xg2Sye6p3yP7vDdrCMDbLj_I/edit?gid=1921994032#gid=1921994032 for perturbation cases to be tested.

### 2.2.1 Perturbation Case 1

In [50]:
sid = adata.obs["Study_slice_id"].astype(str)

adata.obs["DI12_vs_CV"] = np.nan
adata.obs.loc[sid.eq("DI12-KID-0-FT-4"), "DI12_vs_CV"] = "DI12"
adata.obs.loc[sid.str.startswith("CV"), "DI12_vs_CV"] = "CV"

adata.obs["DI12_vs_CV"] = adata.obs["DI12_vs_CV"].astype("category")
adata.obs["DI12_vs_CV"].value_counts(dropna=False)

/tmp/ipykernel_1830773/1722265279.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'DI12' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  adata.obs.loc[sid.eq("DI12-KID-0-FT-4"), "DI12_vs_CV"] = "DI12"


DI12_vs_CV
CV      1081189
NaN      384726
DI12      51979
Name: count, dtype: int64

In [51]:
# Compute differential genes for cell counts (across all cells -> global)
sc.tl.rank_genes_groups(
    adata,
    groupby="DI12_vs_CV",
    groups=["DI12"],
    reference="CV",
    method="wilcoxon",
    pts=True,
    use_raw=False,
    key_added="rank_genes_groups_case1_global_pre_post_cell_counts",
)

# Retrieve differential genes for cell counts
n = 50
diff_genes_df = sc.get.rank_genes_groups_df(adata, group="DI12", key="rank_genes_groups_case1_global_pre_post_cell_counts")
diff_genes_df_top = diff_genes_df.head(n)
diff_genes_df_top[["names", "logfoldchanges", "pvals_adj", "scores"]]
display(diff_genes_df_top)
diff_genes_df_top.to_csv(f"{results_directory_path}/nemokidneyxeniumatlas_top_{n}_diff_genes_case1_global_pre_post_cell_counts.csv")

,names,scores,logfoldchanges,pvals,pvals_adj,pct_nz_group
0,MYH9,160.084213,2.944067,0.0,0.0,0.726717
1,CXCR4,132.442032,3.754869,0.0,0.0,0.418938
2,NR4A1,117.614861,2.496380,0.0,0.0,0.569211
3,THBS1,112.347115,2.797747,0.0,0.0,0.417419
4,PLEC,109.240433,2.128896,0.0,0.0,0.544008
5,EEF1G,107.036507,2.071732,0.0,0.0,0.806999
6,TUBB,105.404610,2.625622,0.0,0.0,0.386368
7,SEPTIN9,95.903107,2.058684,0.0,0.0,0.434214
8,SMAP2,93.913750,3.387367,0.0,0.0,0.291060
9,YWHAZ,93.393776,2.048456,0.0,0.0,0.416611


In [52]:
# Compute differential genes for cell counts (across each niche)
# Loop over niches and run DE within each
min_cells_per_group = 20

summary_rows = []

# Get categories (handles categorical + non-categorical)
cats = adata.obs[niche_key].astype("category").cat.categories

for cat in cats:
    # Cells in this niche AND in either DI12 or CV
    mask = (adata.obs[niche_key] == cat) & (adata.obs["DI12_vs_CV"].isin(["DI12", "CV"]))
    n_total = int(mask.sum())
    if n_total == 0:
        summary_rows.append({"niche": str(cat), "status": "skipped (no DI12/CV cells)", "n_DI12": 0, "n_CV": 0})
        continue

    sub = adata[mask].copy() # copy so we can compute safely without view warnings

    counts = sub.obs["DI12_vs_CV"].value_counts()
    n_di12 = int(counts.get("DI12", 0))
    n_cv   = int(counts.get("CV", 0))

    if n_di12 < min_cells_per_group or n_cv < min_cells_per_group:
        summary_rows.append({
            "niche": str(cat),
            "status": f"skipped (too few cells; min={min_cells_per_group})",
            "n_DI12": n_di12,
            "n_CV": n_cv
        })
        continue

    key = f"rank_genes_groups_case1_niche={cat}_pre_post_cell_counts"

    sc.tl.rank_genes_groups(
        sub,
        groupby="DI12_vs_CV",
        groups=["DI12"],
        reference="CV",
        method="wilcoxon",
        pts=True,
        use_raw=False,
        key_added=key,
    )

    # Retrieve differential genes for cell counts
    n = 50
    diff_genes_df = sc.get.rank_genes_groups_df(sub, group="DI12", key=key)
    diff_genes_df_top = diff_genes_df.head(n)
    diff_genes_df_top[["names", "logfoldchanges", "pvals_adj", "scores"]]
    display(diff_genes_df_top)
    diff_genes_df_top.to_csv(f"{results_directory_path}/nemokidneyxeniumatlas_top_{n}_diff_genes_case1_niche_{cat}_pre_post_cell_counts.csv")
    
    # Store results on the original adata
    adata.uns[key] = sub.uns[key]

    summary_rows.append({
        "niche": str(cat),
        "status": "ok",
        "n_DI12": n_di12,
        "n_CV": n_cv,
        "uns_key": key
    })

# Store a summary table on the AnnData for bookkeeping
adata.uns["DE_case1_pre_post_cell_counts_summary_by_niche"] = pd.DataFrame(summary_rows)
adata.uns["DE_case1_pre_post_cell_counts_summary_by_niche"]

,names,scores,logfoldchanges,pvals,pvals_adj,pct_nz_group
0,NR4A1,70.591805,2.614412,0.000000e+00,0.000000e+00,0.725915
1,MYH9,58.149567,1.777354,0.000000e+00,0.000000e+00,0.743191
2,CXCR4,54.828625,2.939777,0.000000e+00,0.000000e+00,0.386345
3,THBS1,54.413692,2.471376,0.000000e+00,0.000000e+00,0.447982
4,TUBB,39.334911,1.619941,0.000000e+00,0.000000e+00,0.410336
5,YWHAZ,38.423527,1.379470,0.000000e+00,0.000000e+00,0.498001
6,ADAMTS1,36.522076,1.963369,4.949646e-292,1.884292e-289,0.317691
7,CCN2,35.534721,1.871404,1.430635e-276,5.057295e-274,0.322897
8,CYBA,35.521233,1.619179,2.311149e-276,7.625252e-274,0.357676
9,SMAP2,35.453709,2.310190,2.542936e-275,7.865619e-273,0.264957


,names,scores,logfoldchanges,pvals,pvals_adj,pct_nz_group
0,HOXB7,35.662437,2.840188,1.511837e-278,3.741040e-275,0.417697
1,YWHAZ,19.236492,1.450267,1.831955e-82,1.511057e-79,0.339645
2,SOST,18.218763,1.918584,3.663357e-74,2.589994e-71,0.242272
3,THBS1,18.062096,1.497026,6.336999e-73,3.920226e-70,0.312210
4,C4A,15.583218,1.459261,9.466778e-55,4.685108e-52,0.254637
5,NR4A1,15.314026,1.098727,6.162859e-53,2.541666e-50,0.475077
6,P4HB,14.018264,0.943931,1.205268e-44,4.588362e-42,0.363794
7,HNRNPA1L2,12.971551,0.937951,1.774037e-38,5.853141e-36,0.325541
8,MYH9,12.742639,0.759105,3.426095e-37,1.059734e-34,0.511785
9,PRKAR1A,12.459791,0.840640,1.236935e-35,3.600937e-33,0.357805


,names,scores,logfoldchanges,pvals,pvals_adj,pct_nz_group
0,MYH9,43.734844,1.981693,0.000000e+00,0.000000e+00,0.848599
1,TUBB,27.577065,1.969331,2.096714e-167,2.075327e-164,0.362611
2,ANXA2,27.228180,1.545762,3.013520e-163,2.485652e-160,0.553657
3,PLEC,26.328379,1.425019,9.078109e-153,6.418223e-150,0.613807
4,NPNT,24.577225,1.528391,2.213538e-133,1.369350e-130,0.506664
5,PODXL,24.175467,1.477475,4.031158e-129,2.216689e-126,0.673787
6,ACTN4,23.847263,1.318120,1.081073e-125,5.350233e-123,0.621326
7,NR4A1,23.065178,1.573209,1.035877e-117,4.272128e-115,0.388243
8,COL18A1,22.302681,1.407255,3.480120e-110,1.324855e-107,0.405332
9,IGFBP2,21.055374,1.333966,2.041656e-98,7.217253e-96,0.473342


,names,scores,logfoldchanges,pvals,pvals_adj,pct_nz_group
0,TUBB,49.376198,2.214466,0.000000e+00,0.000000e+00,0.453021
1,COL18A1,48.153839,2.136547,0.000000e+00,0.000000e+00,0.467006
2,MYH9,42.761883,1.816269,0.000000e+00,0.000000e+00,0.797400
3,THBS1,39.928429,1.781056,0.000000e+00,0.000000e+00,0.473199
4,ENG,39.448345,1.845039,0.000000e+00,0.000000e+00,0.414059
5,YWHAZ,36.381989,1.593951,8.202925e-290,4.510697e-287,0.425788
6,CYP1B1,35.716808,1.731401,2.168021e-279,1.072954e-276,0.368741
7,ATP5F1A,34.894531,1.478862,8.999039e-267,4.048749e-264,0.467375
8,HNRNPA1L2,34.183445,1.434171,4.261004e-256,1.757309e-253,0.476849
9,CALM3,32.885380,1.664464,3.556660e-237,1.353993e-234,0.331666


,names,scores,logfoldchanges,pvals,pvals_adj,pct_nz_group
0,HOXB7,14.110112,3.401411,3.290521e-45,1.628479e-41,0.438655
1,ACSL4,11.531609,3.233955,9.141961e-31,1.131089e-27,0.347899
2,EEF1G,7.924186,1.393013,2.296465e-15,1.623601e-12,0.621849
3,YWHAZ,7.899892,2.266897,2.791443e-15,1.726856e-12,0.280672
4,ADAMTS1,7.689108,2.799689,1.481638e-14,8.147364e-12,0.236975
5,MYH9,7.630519,1.490531,2.338100e-14,1.157126e-11,0.433613
6,HNRNPA1L2,7.302888,1.691313,2.816556e-13,1.072241e-10,0.332773
7,SDC4,6.963958,1.284530,3.308437e-12,1.023341e-09,0.440336
8,PAPPA2,6.798177,3.347517,1.059514e-11,3.084433e-09,0.188235
9,ITGB6,6.444923,2.944879,1.156588e-10,3.179976e-08,0.188235


,names,scores,logfoldchanges,pvals,pvals_adj,pct_nz_group
0,HOXB7,55.140762,5.116326,0.000000e+00,0.000000e+00,0.718911
1,SOST,25.096365,2.647216,5.449179e-139,1.348399e-135,0.441133
2,KCNJ15,21.670559,1.701075,3.890011e-104,3.850332e-101,0.571376
3,PDZK1,19.089157,1.456277,3.107402e-81,2.563089e-78,0.552980
4,SLC22A8,14.533227,1.351021,7.462082e-48,4.616231e-45,0.397719
5,APP,12.695014,1.031910,6.301830e-37,2.227697e-34,0.443341
6,THY1,12.364964,1.523468,4.043904e-35,1.334219e-32,0.280721
7,ACE2,11.267284,1.313449,1.903503e-29,5.887774e-27,0.293598
8,ATP5F1A,11.175601,0.854259,5.368567e-29,1.562885e-26,0.459161
9,ENPEP,10.875715,1.089371,1.504717e-27,3.919392e-25,0.327079


,niche,status,n_DI12,n_CV,uns_key
0,Distal nephron neighborhood,ok,13255,155325,rank_genes_groups_case1_niche=Distal nephron n...
1,Fibrovascular neighborhood,ok,5176,161046,rank_genes_groups_case1_niche=Fibrovascular ne...
2,Glomerular neighborhood,ok,5852,105607,rank_genes_groups_case1_niche=Glomerular neigh...
3,Injured proximal nephron neighborhood,ok,24383,21174,rank_genes_groups_case1_niche=Injured proximal...
4,Loop of Henle neighborhood,ok,595,141938,rank_genes_groups_case1_niche=Loop of Henle ne...
5,Proximal nephron neighborhood,ok,2718,496099,rank_genes_groups_case1_niche=Proximal nephron...


In [ ]:
# Compute differential genes for neighborhood counts
adata.X = adata.layers['lognorm_neighborhood'].copy()
sc.tl.rank_genes_groups(
    adata,
    groupby="DI12_vs_CV",
    groups=["DI12"],
    reference="CV",
    method="wilcoxon",
    pts=True,
    use_raw=False,
    key_added="rank_genes_groups_case1_global_pre_post_neighborhood_counts",
)

# Retrieve differential genes for neighborhood counts
n = 50
diff_genes_df = sc.get.rank_genes_groups_df(adata, group="DI12", key="rank_genes_groups_case1_global_pre_post_neighborhood_counts")
diff_genes_df_top = diff_genes_df.head(n)
diff_genes_df_top[["names", "logfoldchanges", "pvals_adj", "scores"]]
display(diff_genes_df_top)
diff_genes_df_top.to_csv(f"{results_directory_path}/nemokidneyxeniumatlas_top_{n}_diff_genes_case1_pre_post_neighborhood_counts.csv")

In [ ]:
# Compute differential genes for neighborhood counts (across each niche)
# Loop over niches and run DE within each
min_cells_per_group = 20

summary_rows = []

# Get categories (handles categorical + non-categorical)
cats = adata.obs[niche_key].astype("category").cat.categories

for cat in cats:
    # Cells in this niche AND in either DI12 or CV
    mask = (adata.obs[niche_key] == cat) & (adata.obs["DI12_vs_CV"].isin(["DI12", "CV"]))
    n_total = int(mask.sum())
    if n_total == 0:
        summary_rows.append({"niche": str(cat), "status": "skipped (no DI12/CV cells)", "n_DI12": 0, "n_CV": 0})
        continue

    sub = adata[mask].copy() # copy so we can compute safely without view warnings

    counts = sub.obs["DI12_vs_CV"].value_counts()
    n_di12 = int(counts.get("DI12", 0))
    n_cv   = int(counts.get("CV", 0))

    if n_di12 < min_cells_per_group or n_cv < min_cells_per_group:
        summary_rows.append({
            "niche": str(cat),
            "status": f"skipped (too few cells; min={min_cells_per_group})",
            "n_DI12": n_di12,
            "n_CV": n_cv
        })
        continue

    key = f"rank_genes_groups_case1_niche={cat}_pre_post_neighborhood_counts"

    sc.tl.rank_genes_groups(
        sub,
        groupby="DI12_vs_CV",
        groups=["DI12"],
        reference="CV",
        method="wilcoxon",
        pts=True,
        use_raw=False,
        key_added=key,
    )

    # Retrieve differential genes for cell counts
    n = 50
    diff_genes_df = sc.get.rank_genes_groups_df(sub, group="DI12", key=key)
    diff_genes_df_top = diff_genes_df.head(n)
    diff_genes_df_top[["names", "logfoldchanges", "pvals_adj", "scores"]]
    display(diff_genes_df_top)
    diff_genes_df_top.to_csv(f"{results_directory_path}/nemokidneyxeniumatlas_top_{n}_diff_genes_case1_niche_{cat}_pre_post_neighborhood_counts.csv")
    
    # Store results on the original adata
    adata.uns[key] = sub.uns[key]

    summary_rows.append({
        "niche": str(cat),
        "status": "ok",
        "n_DI12": n_di12,
        "n_CV": n_cv,
        "uns_key": key
    })

# Store a summary table on the AnnData for bookkeeping
adata.uns["DE_case1_pre_post_neighborhood_counts_summary_by_niche"] = pd.DataFrame(summary_rows)
adata.uns["DE_case1_pre_post_neighborhood_counts_summary_by_niche"]

In [ ]:
adata.uns

In [ ]:
# Write AnnData to disk
adata.write(f"{kidney_perturbation_folder_path}/nemokidneyxeniumatlas_annotated_processed.h5ad")

### 2.2.2 Perturbation Case 2

In [ ]:
sid = adata.obs["Study_slice_id"].astype(str)

adata.obs["DI10_vs_CV"] = np.nan
adata.obs.loc[sid.eq("DI10-KID-0-FT-5"), "DI10_vs_CV"] = "DI10"
adata.obs.loc[sid.str.startswith("CV"), "DI10_vs_CV"] = "CV"

adata.obs["DI10_vs_CV"] = adata.obs["DI10_vs_CV"].astype("category")
adata.obs["DI10_vs_CV"].value_counts(dropna=False)

## Tokenize Data

In [ ]:
dataset = tokenize_adata(
    adata,
    model_folder_path=model_folder_path,
    cache_directory_path=cache_directory_path,             
    nproc=4,
    processing_mode='parallel',
    add_neigh_cell_ids=True,
    use_generator=True,
    keep_in_memory=False)

## Load Tokenized Data

In [ ]:
batch_name = 'CV1-KID-0-FT-5-S5'
tokenized_dataset_name = f'tokenized_{batch_name}'

tokenized_data_folder_path = '/nfs/team361/dj17/NEMO_DJ/KidneyToxicity/NEMO_UpdatedTokenisationandEmbedding_November22nd25/tokenised_xenium_common_qc'
load_dataset_path = f'{tokenized_data_folder_path}/{tokenized_dataset_name}'
dataset = load_from_disk(load_dataset_path)
dataset

In [ ]:
adata_subset = adata[adata.obs[batch_key] == batch_name]

In [ ]:
# Add dummy batch value
dataset = dataset.add_column("batch_value", [[999]] * len(dataset))

## Retrieve Embeddings

In [ ]:
output_embed = embed_dataset(
    dataset=dataset,
    model_folder_path=model_folder_path,
    emb_layer=emb_layer,
    agg_excluded_tokens=agg_excluded_tokens,
    top_k=top_k,
    batch_size=batch_size,
    pin_memory=pin_memory,
    num_workers=num_workers,
    ignore_spc_tokens=True)

## Perform Perturbations

## Perturbation 1

In [ ]:
from collections import defaultdict

In [ ]:
index = defaultdict(list)
for row in df.itertuples(index=False):
    index[row.perturbed_cell_id].append(row._asdict())

In [ ]:
from pathlib import Path
import pickle

In [ ]:
# Load token dictionary
with open(Path(model_folder_path) / "token_dictionary.pkl", "rb") as f:
    token_dict = pickle.load(f)

In [ ]:
Pre treatment, any DIAMOND, post-treatment, Study_slice_id = DI12-KID-0-FT-4

In [ ]:
adata.obs

In [ ]:
adata.var.loc['PDCD1']

In [ ]:
CTLA4 - ENSG00000163599
PDCD1 - ENSG00000188389

In [ ]:
len(adata_subset.obs['cell_id'].tolist() * 4)

In [ ]:
# Define perturbations
perturbed_cell_id = adata_subset.obs['cell_id'].tolist()
perturbed_ensembl_id = ['ENSG00000163599', 'ENSG00000163599', 'ENSG00000188389', 'ENSG00000188389'] * len(perturbed_cell_id) # ['all'] * len(perturbed_cell_id)
perturbation_target = ['cell', 'neighborhood', 'cell', 'neighborhood'] * len(perturbed_cell_id) # ['neighborhood'] * len(perturbed_cell_id)
perturbation_type = ['knockout', 'knockout', 'knockout', 'knockout'] * len(perturbed_cell_id) # ['knockout'] * len(perturbed_cell_id)
foldchange = [np.nan, np.nan, np.nan, np.nan] * len(perturbed_cell_id) # [np.nan] * len(perturbed_cell_id)
perturbed_cell_id = perturbed_cell_id * 4

perturb_df =  pd.DataFrame({
    'perturbed_cell_id': perturbed_cell_id,
    'perturbed_ensembl_id': perturbed_ensembl_id,
    'perturbation_target': perturbation_target,
    'perturbation_type': perturbation_type, 
    'foldchange': foldchange
})

In [ ]:
perturb_df = perturb_df.head(3)

In [ ]:
def _build_perturb_index(df: pd.DataFrame) -> dict[str, list[dict]]:
    """
    Turn perturb_df into an index:
        {perturbed_cell_id: [row_as_dict, …]}
    (Using itertuples avoids Python object creation for every column.)
    """
    index = defaultdict(list)
    for row in df.itertuples(index=False):
        index[row.perturbed_cell_id].append(row._asdict())
    return index

def _perturb_batch(
    batch: dict,
    index: dict[str, list[dict]],
    seq_len_cell: int = 256,
) -> dict:
    """
    `batch` is the dictionary of column -> list-of-values
    returned by Hugging-Face when batched=True.
    Modify the tensors in-place (cheap) and return the same dict.
    """
    B = len(batch["cell_ids"])          # batch size
    for b in range(B):
        cell_ids: list[str] = list(dict.fromkeys(batch["cell_ids"][b]))
        # Fast reject: does this example touch any perturbed cell at all?
        if not any(cid in index for cid in cell_ids):
            continue

        # Handle index-cell (first) and neighbourhood (rest) separately
        for cid in cell_ids:
            for row in index.get(cid, []):
                is_index_cell = (cid == cell_ids[0])
                if row["perturbation_target"] == "cell" and not is_index_cell:
                    continue
                if row["perturbation_target"] == "neighborhood" and is_index_cell:
                    continue

                gene_tokens = batch["gene_tokens"][b]
                gene_expr   = batch["gene_expr"][b]

                # --- choose which token positions to perturb -------------
                if row["perturbed_gene_token"] == "all":
                    if is_index_cell:
                        idx = slice(0, seq_len_cell)
                    else:
                        idx = slice(seq_len_cell, None)
                else:
                    token_id = row["perturbed_gene_token"]
                    token_slice = (
                        gene_tokens[:seq_len_cell]
                        if is_index_cell else
                        gene_tokens[seq_len_cell:]
                    )
                    rel_idx = torch.nonzero(token_slice == token_id, as_tuple=True)[0]
                    offset  = 0 if is_index_cell else seq_len_cell
                    idx     = rel_idx + offset
                # ---------------------------------------------------------

                if row["perturbation_type"] == "knockout":
                    gene_expr[idx]   = 0.0
                elif row["perturbation_type"] == "foldchange":
                    gene_expr[idx] *= row["foldchange"]
                else:
                    raise ValueError(f"Bad perturbation_type: {row['perturbation_type']}")

    return batch

In [ ]:
from functools import partial

In [ ]:
seq_len_cell = 256

In [ ]:
perturb_index

In [ ]:
    # 3) Build an index for O(1) lookup
    perturb_index = _build_perturb_index(perturb_df)
    # 4) Partial-apply so the mapper sees only one argument
    perturb_fn = partial(
        _perturb_batch,
        index=perturb_index,
        seq_len_cell=seq_len_cell,
    )

In [ ]:
import time

start = time.time()

# --- code you want to time ---
perturbed_dataset = perturb_dataset(
    dataset=dataset,
    perturb_df=perturb_df,
    nproc=1,
    model_folder_path=model_folder_path)
# ----------------------------

elapsed_minutes = (time.time() - start) / 60
print(f"Runtime: {elapsed_minutes:.2f} minutes")

In [ ]:
perturbed_output_embed = embed_dataset(
    dataset=perturbed_dataset,
    model_folder_path=model_folder_path,
    emb_layer=emb_layer,
    cell_gene_ids=cell_gene_ids,
    neighborhood_gene_ids=neighborhood_gene_ids,
    agg_excluded_tokens=agg_excluded_tokens,
    top_k=top_k,
    batch_size=batch_size,
    pin_memory=pin_memory,
    num_workers=num_workers)

In [ ]:
adata.obsm['perturbed_cell_emb'] = perturbed_output_embed['cell_emb']
adata.obsm['perturbed_neighborhood_emb'] = perturbed_output_embed['neighborhood_emb']

In [ ]:
label = cell_type_key

# Neighborhood embedding UMAP
sc.pp.neighbors(adata,
                n_neighbors=15,
                use_rep='perturbed_neighborhood_emb',
                key_added='perturbed_neighborhood')
sc.tl.umap(adata,
           neighbors_key='perturbed_neighborhood',
           key_added='perturbed_neighborhood_umap')
adata.obsm['X_umap'] = adata.obsm['perturbed_neighborhood_umap']
sc.pl.umap(adata,
           neighbors_key='perturbed_neighborhood',
           color=label)

# Cell embedding UMAP
sc.pp.neighbors(adata,
                n_neighbors=15,
                use_rep='perturbed_cell_emb',
                key_added='perturbed_cell')
sc.tl.umap(adata,
           neighbors_key='perturbed_cell',
           key_added='perturbed_cell_umap')
adata.obsm['X_umap'] = adata.obsm['perturbed_cell_umap']
sc.pl.umap(adata,
           neighbors_key='perturbed_cell',
           color=label)